# Conjunto de factoid con la respuesta fuera del grafo


In [1]:
import json
import re
import sys

import pandas as pd

sys.path.insert(0, "../../../graphrag")
from entidades.linkear_entidades import Linkeador, menciones_de

BIOASQ = "../../preguntas/BioASQ-training14b/training14b.json"
EXTRACCIONES = "extracciones.jsonl"
TIPOS_DE_NODO = ["Gene", "Disease", "Drug"]
#el nombre de la tarea en los scripts de la carpeta de arriba
TAREA = "factoid_sin_oro"
#tantas como la muestra de factoid con la respuesta en el grafo (111, armar_conjuntos.ipynb)
PREGUNTAS = 111
SEED = 42
pd.set_option("display.width", 250)
pd.set_option("display.max_colwidth", 100)

## Carga de las preguntas de factoid

In [2]:
with open(BIOASQ) as archivo:
    todas = json.load(archivo)["questions"]

preguntas = pd.DataFrame([{"id": p["id"], "body": p["body"], "oro": [p["exact_answer"]], "ideal_answer": p["ideal_answer"]}
                          for p in todas if p["type"] == "factoid"])
len(preguntas)

1695

## entidades de la pregunta

Igual que en armar_conjuntos.ipynb: las menciones guardadas de Qwen, cada una enlazada con linkear_mencion.

In [3]:
extracciones = {}
with open(EXTRACCIONES) as archivo:
    for linea in archivo:
        fila = json.loads(linea)
        extracciones[fila["id_bioasq"]] = fila["respuesta"]
assert preguntas["id"].isin(extracciones).all()

linkeador = Linkeador()

entidades, enlaces, sin_enlazar, menciones = [], [], [], []
for pregunta in preguntas.itertuples():
    entidades_de_la_pregunta, enlaces_de_la_pregunta, menciones_sin_nodo, textos = [], [], [], []
    for mencion in menciones_de(extracciones[pregunta.id]):
        textos.append(mencion.texto)
        enlace = linkeador.linkear_mencion(mencion)
        if enlace is None:
            menciones_sin_nodo.append(mencion.texto)
            continue
        label, ids, paso = enlace
        enlaces_de_la_pregunta.append([mencion.texto, label, ids, paso])
        for id_nodo in ids:
            if [label, id_nodo] not in entidades_de_la_pregunta:
                entidades_de_la_pregunta.append([label, id_nodo])
    entidades.append(entidades_de_la_pregunta)
    enlaces.append(enlaces_de_la_pregunta)
    sin_enlazar.append(menciones_sin_nodo)
    menciones.append(textos)
preguntas["menciones"] = menciones
preguntas["entidades"] = entidades
preguntas["enlaces"] = enlaces
preguntas["sin_enlazar"] = sin_enlazar
preguntas["con_mencion"] = preguntas["menciones"].str.len() > 0
preguntas["todas_enlazan"] = preguntas["con_mencion"] & (preguntas["sin_enlazar"].str.len() == 0)

## oro y cuanto se va perdiendo

El oro se enlaza como en armar_conjuntos.ipynb.

In [4]:
preguntas["oro_con_nodo"] = [any(linkeador.exacto(tipo, [sinonimo]) for sinonimo in pregunta.oro[0] for tipo in TIPOS_DE_NODO)
                             for pregunta in preguntas.itertuples()]

CONDICIONES = [("1. Qwen marca alguna mención", preguntas["con_mencion"]),
               ("2. todas las menciones enlazan", preguntas["todas_enlazan"]),
               ("3. la respuesta no tiene nodo", ~preguntas["oro_con_nodo"])]
quedan = pd.Series(True, index=preguntas.index)
filas = {"preguntas de factoid en 14b": len(preguntas)}
for nombre, condicion in CONDICIONES:
    quedan = quedan & condicion
    filas[nombre] = quedan.sum()
filas["(la respuesta con nodo: conjunto A antes de la condición 4)"] = (preguntas["todas_enlazan"] & preguntas["oro_con_nodo"]).sum()
entran = preguntas[quedan]
pd.Series(filas).to_frame("preguntas")

,preguntas
preguntas de factoid en 14b,1695
1. Qwen marca alguna mención,1241
2. todas las menciones enlazan,525
3. la respuesta no tiene nodo,395
(la respuesta con nodo: conjunto A antes de la condición 4),130


## Ejemplos para revisar

Algunos sorteados, para ver a ojo cuánto falta en el grafo y cuánto es falla del enlace.

In [5]:
for pregunta in entran.sample(n=15, random_state=SEED).itertuples():
    print(f"- {pregunta.body}")
    print(f"    respuesta: {pregunta.oro[0]}    entidades de la pregunta: {[e[0] for e in pregunta.enlaces]}")

- In which process Src, Cortactin and MT1-MMP are playing an essential role?
    respuesta: ['Src, Cortactin and MT1-MMP play an essential role in the formation and function of invadopodia.']    entidades de la pregunta: ['Src', 'Cortactin', 'MT1-MMP']
- What is the median survival time for glioblastoma patients with standard treatment?
    respuesta: ['15 months', '12-15 months', '12 to 15 months', '14-16 months', '14 to 15 months', '14-15 months', '14.5 months', '14.6 months']    entidades de la pregunta: ['glioblastoma']
- What is the chromosomal location of the LDL receptor gene associated with autosomal dominant Familial Hypercholesterolemia?
    respuesta: ['short arm of chromosome 19']    entidades de la pregunta: ['LDL receptor gene', 'autosomal dominant Familial Hypercholesterolemia']
- What causes Katayama Fever?
    respuesta: ['Schistosoma spp']    entidades de la pregunta: ['Katayama Fever']
- What is the cause of Fibromyalgia?
    respuesta: ['unknown']    entidades de la

In [6]:
nombre_de = linkeador.nombre_de

consultas = {}
for pregunta in entran.sort_values("id").itertuples():
    label, id_nodo = pregunta.entidades[0]
    base = re.sub(r"[^a-z0-9]+", "_", nombre_de[(label, id_nodo)].lower()).strip("_")
    nombre = base + "_" + pregunta.id[-6:]
    if nombre in consultas:
        nombre = base + "_" + pregunta.id
    assert nombre not in consultas, f"nombre repetido: {nombre}"
    consultas[nombre] = {"pregunta": pregunta.body, "entidades": pregunta.entidades, "id_bioasq": pregunta.id,
                         "tarea": "factoid", "enlaces": pregunta.enlaces, "sin_enlazar": pregunta.sin_enlazar,
                         "oro": pregunta.oro, "oro_nodos": [[]], "ideal_answer": pregunta.ideal_answer}
with open(f"consultas_{TAREA}.json", "w") as archivo:
    json.dump(consultas, archivo, indent=1, ensure_ascii=False)

nombres = pd.Series(sorted(consultas))
elegidos = nombres.sample(n=min(PREGUNTAS, len(nombres)), random_state=SEED).sort_values()
muestra = {nombre: consultas[nombre] for nombre in elegidos}
with open(f"muestra_{TAREA}.json", "w") as archivo:
    json.dump(muestra, archivo, indent=1, ensure_ascii=False)

pd.DataFrame([{"consulta": consulta, "id_bioasq": datos["id_bioasq"], "pregunta": datos["pregunta"],
               "menciones": " ; ".join(f"{e[0]} -> {' | '.join(nombre_de[(e[1], i)] for i in e[2])} [{e[3]}]" for e in datos["enlaces"]),
               "entidades": " ; ".join(f"{label}:{i}" for label, i in datos["entidades"]),
               "respuesta": " | ".join(datos["oro"][0]), "en_muestra": consulta in muestra}
              for consulta, datos in consultas.items()]).to_csv(f"preguntas_{TAREA}.csv", index=False)
print(f"consultas_{TAREA}.json", len(consultas), "   muestra", len(muestra))

consultas_factoid_sin_oro.json 395    muestra 111
